# Flexynesis — Inference Workshop

학습된 모델을 불러와 예측, 임베딩 추출, feature importance까지 실행하는 노트북.

> 모델 학습은 `flexynesis_workshop.ipynb` 참고


In [ ]:
import os
import flexynesis
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("flexynesis:", flexynesis.__version__)
print("CUDA available:", torch.cuda.is_available())


## 1. 데이터 로드

In [ ]:
DATA_DIR = "/data01/storage/flexynesis_workshop/brca_metabric_processed"
MINI_DIR  = "/data01/storage/flexynesis_workshop/brca_mini"

USE_MINI = True
PATH = MINI_DIR if USE_MINI else DATA_DIR
print(f"사용 데이터: {PATH}")

data_importer = flexynesis.data.DataImporter(
    path=PATH,
    data_types=['gex', 'cna'],
    top_percentile=10, min_features=50,
    correlation_threshold=0.9, variance_threshold=0.8,
    na_threshold=0.1, log_transform=False,
    covariates=None, downsample=0, restrict_to_features=None,
)
train_dataset, test_dataset = data_importer.import_data()
print("Test samples:", len(test_dataset.samples))


## 2. 모델 불러오기

In [ ]:
MODEL_DIR = "./models"

model_dp    = torch.load(f"{MODEL_DIR}/DirectPred_clf.pth",    weights_only=False)
model_surv  = torch.load(f"{MODEL_DIR}/DirectPred_surv.pth",   weights_only=False)
model_vae   = torch.load(f"{MODEL_DIR}/supervised_vae_clf.pth",weights_only=False)
model_unsup = torch.load(f"{MODEL_DIR}/supervised_vae_unsup.pth", weights_only=False)
model_tri   = torch.load(f"{MODEL_DIR}/MultiTripletNetwork.pth",weights_only=False)
model_cross = torch.load(f"{MODEL_DIR}/CrossModalPred.pth",    weights_only=False)

print("모델 로드 완료")


In [ ]:
# GNN은 별도 데이터셋 필요
from flexynesis.data import STRING, MultiOmicDatasetNW

data_importer_gnn = flexynesis.data.DataImporter(
    path=PATH, data_types=['gex'],
    top_percentile=10, min_features=50,
    correlation_threshold=0.9, variance_threshold=0.8,
    na_threshold=0.1, log_transform=False,
    covariates=None, downsample=0, restrict_to_features=None,
)
_, test_gnn_raw = data_importer_gnn.import_data()

string_obj = STRING(organism=9606, node_name='gene_name')
test_gnn   = MultiOmicDatasetNW(test_gnn_raw, string_obj.graph_df, modality_order=['gex'])

model_gnn = torch.load(f"{MODEL_DIR}/GNN.pth", weights_only=False)
print("GNN 로드 완료")


## 3. 예측

In [ ]:
labels = [test_dataset.label_mappings['CLAUDIN_SUBTYPE'][x]
          for x in test_dataset.ann['CLAUDIN_SUBTYPE'].numpy()]

all_metrics = []

for model, name, dataset in [
    (model_dp,    'DirectPred',          test_dataset),
    (model_vae,   'supervised_vae',      test_dataset),
    (model_tri,   'MultiTripletNetwork', test_dataset),
    (model_gnn,   'GNN',                 test_gnn),
    (model_cross, 'CrossModalPred',      test_dataset),
]:
    m = flexynesis.utils.evaluate_wrapper(
        method=name,
        y_pred_dict=model.predict(dataset),
        dataset=dataset,
    )
    all_metrics.append(m)
    print(f"{name}: done")


In [ ]:
# 생존분석 평가
metrics_surv = flexynesis.utils.evaluate_wrapper(
    method='DirectPred_survival',
    y_pred_dict=model_surv.predict(test_dataset),
    dataset=test_dataset,
    surv_event_var=model_surv.surv_event_var,
    surv_time_var=model_surv.surv_time_var,
)
print("C-index:", metrics_surv[metrics_surv['metric'] == 'cindex']['value'].values)


## 4. 모델 비교

In [ ]:
clf_metrics = [m for m in all_metrics if 'balanced_acc' in m['metric'].values]
pivot = pd.concat(clf_metrics).pivot_table(index='method', columns='metric', values='value')
pivot = pivot[['balanced_acc', 'f1_score', 'average_auroc', 'average_aupr']]
pivot.sort_values('average_auroc', ascending=False)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
pivot[['balanced_acc', 'f1_score', 'average_auroc']].plot(kind='bar', ax=ax, rot=0)
ax.set_title('Model Comparison — CLAUDIN_SUBTYPE')
ax.set_ylim(0, 1)
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5)
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()


## 5. Embedding (UMAP)

In [ ]:
models_to_plot = [
    (model_dp,    'DirectPred',          test_dataset, labels),
    (model_vae,   'supervised_vae',      test_dataset, labels),
    (model_tri,   'MultiTripletNetwork', test_dataset,
     [test_dataset.label_mappings['CLAUDIN_SUBTYPE'][x]
      for x in test_dataset.ann['CLAUDIN_SUBTYPE'].numpy()]),
    (model_gnn,   'GNN', test_gnn,
     [test_gnn.label_mappings['CLAUDIN_SUBTYPE'][x]
      for x in test_gnn.ann['CLAUDIN_SUBTYPE'].numpy()]),
]

for model, name, dataset, lbls in models_to_plot:
    print(f"\n{name}")
    E = model.transform(dataset)
    flexynesis.utils.plot_dim_reduced(E, lbls, method='umap', color_type='categorical', title=name)


## 6. Feature Importance

In [ ]:
# Integrated Gradients로 중요 유전자 발굴
model_dp.compute_feature_importance(train_dataset, 'CLAUDIN_SUBTYPE')

top_features = flexynesis.utils.get_important_features(
    model_dp, var='CLAUDIN_SUBTYPE', top=20
)
top_features


## 7. Kaplan-Meier (생존분석)

In [ ]:
outputs = model_surv.predict(test_dataset)['OS_STATUS']
if outputs.ndim > 1:
    outputs = outputs[:, 0]
risk_scores = np.exp(outputs.flatten())
groups = np.digitize(risk_scores, np.quantile(risk_scores, [0.5]))

durations = test_dataset.ann['OS_MONTHS'].float()
events    = test_dataset.ann['OS_STATUS'].float()
valid_t   = ~torch.isnan(durations) & ~torch.isnan(events)

flexynesis.utils.plot_kaplan_meier_curves(
    durations[valid_t], events[valid_t], groups[valid_t.numpy()]
)
